In [15]:
def edit_distance(p, t):
    """
    Compute the minimum edit distance between pattern p and any substring of text t.
    Approximate matching version (Coursera DNA Sequencing HW3).
    """
    m = len(p)
    n = len(t)

    # row 0: distance between empty pattern and any prefix of t
    # for approximate matching, this should be all zeros
    prev = [0] * (n + 1)
    curr = [0] * (n + 1)

    for i in range(1, m + 1):
        curr[0] = i  # cost of deleting all characters of p[0:i]

        for j in range(1, n + 1):
            if p[i - 1] == t[j - 1]:
                cost = 0
            else:
                cost = 1

            curr[j] = min(
                prev[j] + 1,        # deletion
                curr[j - 1] + 1,    # insertion
                prev[j - 1] + cost  # substitution
            )

        prev, curr = curr, prev  # swap rows

    # after filling all rows, prev is the last row: distances between p and all prefixes of t
    # minimum over this row = best match of p against any substring of t
    return min(prev)

def overlap(a, b, min_length=3):
    """
    Return length of the longest suffix of 'a' matching
    a prefix of 'b' that is at least 'min_length' characters long.
    If no such overlap exists, return 0.
    """
    start = 0  # start all the way at the left of b

    while True:
        start = a.find(b[:min_length], start)
        if start == -1:
            return 0
        # Found a potential match
        if b.startswith(a[start:]):
            return len(a) - start
        start += 1


def build_overlap_graph(reads, k):
    """
    Build an overlap graph for the given reads.
    Return a list of edges (a, b) where overlap(a, b) >= k.
    """
    edges = []
    for a in reads:
        for b in reads:
            if a != b:
                olen = overlap(a, b, k)
                if olen >= k:
                    edges.append((a, b))
    return edges

def overlap_all_pairs(reads, k):
    results = []
    for a in reads:
        for b in reads:
            if a != b:
                olen = overlap(a, b, k)
                if olen >= k:
                    results.append((a, b))
    return results

In [1]:
def readGenome(filename):
    genome = ''
    with open(filename, 'r') as f:
        for line in f:
            # ignore header line with genome information
            if not line[0] == '>':
                genome += line.rstrip()
    return genome

In [14]:
chr1_genome = readGenome('chr1.GRCh38.excerpt.fasta')

In [16]:
pattern = "GCTGATCGATCGTACG"
edit_distance(pattern, chr1_genome)

3

In [17]:
pattern = "GATTTACCAGATTGAG"
edit_distance(pattern, chr1_genome)

2

In [6]:
def readFastq(filename):
    sequences = []
    qualities = []
    with open(filename) as fh:
        while True:
            fh.readline()  # skip name line
            seq = fh.readline().rstrip()  # read base sequence
            fh.readline()  # skip placeholder line
            qual = fh.readline().rstrip() # base quality line
            if len(seq) == 0:
                break
            sequences.append(seq)
            qualities.append(qual)
    return sequences, qualities

In [7]:
overlap_reads = readFastq("ERR266411_1.for_asm.fastq")

In [8]:
sequences = overlap_reads[0]
sequences[:10]

['TAAACAAGCAGTAGTAATTCCTGCTTTATCAAGATAATTTTTCGACTCATCAGAAATATCCGAAAGTGTTAACTTCTGCGTCATGGAAGCGATAAAACTC',
 'AACAAGCAGTAGTAATTCCTGCTTTATCAAGATAATTTTTCGACTCATCAGAAATATACGAAAGTGTTAACTTCTGCGTCATGGACACGAAAAAACTCCC',
 'AACAAGCAGTAGTAATTCCTGCTTTATCAAGATAATTTTTCGACTCATCAGAAATATCCGAAAGTGTTAACTTCTGCGTCATGGAAGCGATAAAACTCTG',
 'AGCCGACGTTTTGGCGGCGCAACCTGTGACGACAAATCTGCTCAAATTTATGCGCGCTTCGATAAAAATGATTGGCGTATCCAACCTGCAGAGTTTTATC',
 'GACAAATCTGCTCAAATTTATGCGCGCTTCGATAAAAATGATTGGCGTATCCAACCTGCAGAGTTTTATCGCTTCCATGACGCAGAAGTTAACACTTTCG',
 'CTGTAGCCGACGTTTTGGCGGCGCAACCTGTGACGACAAATCTGCTCAAATTTATGCGCGCTTCGATAAAAATGATTGGCGTATCCAACCTGCAGAGTTT',
 'CTGTGACGACAAATCTGCTCAAATTTATGCGCGCTTCGATAAAAATGATTGGCGTATCCAACCTGCAGAGTTTTATCGCTTCCATGACGCAGAAGTTAAC',
 'CAAATCTGCTCAAATTTATGCGCGCTTCGATAAAAATGATTGGCGTATCCAACCTGCAGAGTTTTATCGCTTCCATGACGCAGAAGTTAACACTTTCGGA',
 'GTAAACAAGCAGTAGTAATTCCTGCTTTATCAAGATAATTTTTCGACTCATCAGCAATATCCGAAAGAGTTAACTTTTGCGTCATGGAAGCGATAAAACC',
 'GTAAACAAGCAGTAGTAATTCCTGCTTTATCAAGATAATTTTTCGACTCATCA

In [11]:
def find_overlaps(reads, k):
    # Step 1: build k-mer index
    index = {}
    for read in reads:
        for i in range(len(read) - k + 1):
            kmer = read[i:i+k]
            index.setdefault(kmer, set()).add(read)

    # Step 2: find overlaps
    overlaps = []
    for a in reads:
        suffix = a[-k:]
        if suffix in index:
            for b in index[suffix]:
                if a != b:
                    overlaps.append((a, b))
    return overlaps

In [13]:
def build_kmer_index(reads, k):
    index = {}
    for read in reads:
        for i in range(len(read) - k + 1):
            kmer = read[i:i+k]
            index.setdefault(kmer, set()).add(read)
    return index

def find_overlaps(reads, k):
    index = build_kmer_index(reads, k)
    edges = []

    for a in reads:
        suffix = a[-k:]
        if suffix in index:
            for b in index[suffix]:
                if a != b:
                    # exact match guaranteed by k-mer, but still check full overlap
                    if overlap(a, b, k) >= k:
                        edges.append((a, b))
    return edges

# 使用 k=30
edges = find_overlaps(sequences, 30)

# Q1: 边的数量
print("Number of edges:", len(edges))

# Q2: 有 outgoing edge 的节点数量
print("Nodes with outgoing edges:", len(set(a for (a, b) in edges)))

Number of edges: 904746
Nodes with outgoing edges: 7161


In [12]:
find_overlaps(sequences, 30)

[('...'),
 ...]